阶段1：构造模型

在票价确定的前提下，预测各航段的客流量（已完成）。

阶段2：解析过程

根据客流信息分配舱位，以确定总收益。
客舱分为两类：短途类（AB或BC，称为S类）和长途类（AC，称为L类），总容量为C。

根据是否满座进行分类：

若不满座（S+L≤C），则按S、L和C的预测比例分配舱位，以实现收益最大化。
若满座（S+L>C），则需根据销售量变化预测进行取舍，并作如下判断：
若票价AB+BC>AC，则优先将舱位拆分为AB和BC段销售，S的值取min(AB, BC)，其余舱位分配给L。
若票价AB+BC<AC，则优先分配舱位给AC段，剩余舱位分配给AB和BC段。

阶段3：求总收益最大值

以票价为自变量，总收益为因变量，绘制总收益曲线，并求出总收益的最大值。

## 前期工作

在前期的工作中应该已经根据某获取了某个模型，输入航班，票价等参数后可以获取客流量pax信息

如何将模型保存，然后我在这个jupyter文件中使用？

## 加载模型

In [1]:
import xgboost as xgb
import pandas as pd
from sklearn.model_selection import train_test_split

# 加载模型
model = xgb.XGBRegressor()
model.load_model("xgboost_model.json")

print("模型已加载")

模型已加载


In [2]:
# 加载数据
data = pd.read_csv('../../data-hh/my/202410221653-processed.csv', dtype={'aircraft': str})


In [3]:
import joblib
import pandas as pd

# 定义需要编码的分类特征
categorical_columns = ['flt_no', 'bd_type', 'aircraft', 'a', 'b', 'c', 'from', 'to']

# 加载每个编码器并对测试数据进行编码
for col in categorical_columns:
    # 指定编码器保存的路径
    encoder_path = f"../../data-hh/my/encoder/{col}_encoder.pkl"
    
    # 加载编码器
    le = joblib.load(encoder_path)
    
    # 对测试数据进行标签编码
    data[col] = le.transform(data[col])
    
    # 打印提示，告知编码器已应用
    print(f"{col} 已完成标签编码")

# 测试数据现在已准备好用于预测
print(data.head())

flt_no 已完成标签编码
bd_type 已完成标签编码
aircraft 已完成标签编码
a 已完成标签编码
b 已完成标签编码
c 已完成标签编码
from 已完成标签编码
to 已完成标签编码
   flt_no  bd_type    cap  aircraft  legs  leg_no  duration  pax    a    b  \
0    5995        3  132.0         1     1       1      1.70  127   28  142   
1    6535        3  132.0         1     1       1      2.28   60  138   28   
2    2640        3  164.0         2     1       1      1.63  161  211   88   
3    2815        3  164.0         2     1       1      1.55   57   85  220   
4    7627        3  194.0         3     1       1      1.70  191  211  188   

   ...  year  month  day  weekday  hour  minute  second  from   to  \
0  ...  2023      1    1        6    13      25       0    28  142   
1  ...  2023      1    1        6    16      15       0   142   28   
2  ...  2023      1    1        6    17       5       0   218   88   
3  ...  2023      1    1        6    19      55       0    88  219   
4  ...  2023      1    1        6    16      30       0   218  188   

    unit

In [4]:
# 定义模型输入特征列
input_features = ['flt_no', 'bd_type', 'cap', 'aircraft', 'legs', 'leg_no', 'duration', 
                  'a', 'b', 'c', 'year', 'month', 'day', 'weekday', 'hour', 'minute', 'second', 
                  'from', 'to', 'unit_price']

# 定义模型预测特征（这里是 'pax'，但在预测时不需要作为输入）
output_feature = 'pax'

# 提取模型输入特征
X_test = data[input_features]

# 使用模型进行预测
predictions = model.predict(X_test)

# 将预测结果添加到测试数据中
data['pax_prediction'] = predictions

# 查看前几行预测结果
print(data[['pax', 'pax_prediction']].head(20))

    pax  pax_prediction
0   127       93.577560
1    60       81.288231
2   161      120.305305
3    57       91.249695
4   191      131.017731
5    29      117.838379
6   152      112.218712
7    45      104.375839
8   124      114.032417
9    31       94.085876
10  182      152.331696
11   60      135.731720
12   56       66.204521
13  113       79.387238
14   26       49.341080
15   23       64.912888
16  101       84.604362
17   26       58.087208
18   88       72.257629
19  127       91.938560


## 座位分配

根据三段客流量按照分配策略进行分配

写一个函数
    
    输入为ab,bc,ac的票价和预测客流量和cap
    输出为ab，bc，ac分配的舱位数和总收益

In [5]:
def load_model_and_encoders():
    """
    加载所有模型和编码器。
    返回：
    - model: 预测所有航段（AB, BC, AC）的单一模型。
    - label_encoders: 一个字典，包含分类特征的 LabelEncoder。
    """
    # 加载XGBoost模型
    model = xgb.XGBRegressor()
    model.load_model("xgboost_model.json")
    
    # 加载编码器
    categorical_columns = ['flt_no', 'bd_type', 'aircraft', 'a', 'b', 'c', 'from', 'to']
    label_encoders = {}
    for col in categorical_columns:
        label_encoders[col] = joblib.load(f"../../data-hh/my/encoder/{col}_encoder.pkl")
    
    return model, label_encoders

In [6]:
def predict_seat_allocation(flt_no, bd_type, cap, aircraft, duration, 
                            a, b, c, year, month, day, weekday, hour, minute, second, 
                            ab_price, bc_price, ac_price):
    """
    根据输入特征，预测航段 AB, BC, AC 的座位分配 (pax)。
    """
    from xgboost import XGBRegressor
    import joblib
    import pandas as pd

    # 加载模型
    model = XGBRegressor()
    model.load_model("xgboost_model.json")

    # 加载编码器
    categorical_columns = ['flt_no', 'bd_type', 'aircraft', 'a', 'b', 'c', 'from', 'to']
    label_encoders = {}
    for col in categorical_columns:
        label_encoders[col] = joblib.load(f"../../data-hh/my/encoder/{col}_encoder.pkl")

    # 定义特征顺序
    input_features = [
        'flt_no', 'bd_type', 'cap', 'aircraft', 'legs', 'leg_no', 'duration',
        'a', 'b', 'c', 'year', 'month', 'day', 'weekday', 'hour', 'minute',
        'second', 'from', 'to', 'unit_price'
    ]

    # 构建基础数据
    legs = 3
    base_data = {
        'flt_no': [flt_no],
        'bd_type': [bd_type],
        'cap': [cap],
        'aircraft': [aircraft],
        'legs': [legs],
        'leg_no': [1],  # 初始设置为 1，后续根据航段进行更新
        'duration': [duration],
        'a': [a],
        'b': [b],
        'c': [c],
        'year': [year],
        'month': [month],
        'day': [day],
        'weekday': [weekday],
        'hour': [hour],
        'minute': [minute],
        'second': [second]
    }
    base_df = pd.DataFrame(base_data)

    # 生成航段数据
    segments = {
        'AB': {'from': a, 'to': b, 'leg_no': 1, 'unit_price': ab_price},
        'BC': {'from': b, 'to': c, 'leg_no': 2, 'unit_price': bc_price},
        'AC': {'from': a, 'to': c, 'leg_no': 3, 'unit_price': ac_price},
    }

    # 存储结果
    results = {}

    for segment, endpoints in segments.items():
        segment_df = base_df.copy()
        segment_df['from'] = endpoints['from']
        segment_df['to'] = endpoints['to']
        segment_df['leg_no'] = endpoints['leg_no']
        segment_df['unit_price'] = endpoints['unit_price']

        # 对分类特征进行标签编码
        for col in categorical_columns:
            le = label_encoders[col]
            try:
                segment_df[col] = le.transform(segment_df[col])
            except KeyError:
                segment_df[col] = le.transform(['unknown'] * len(segment_df))

        # 确保列顺序与训练时一致
        segment_df = segment_df[input_features]

        # 使用模型进行预测
        predictions = model.predict(segment_df)  # 直接使用 DataFrame
        results[f"{segment}_PAX"] = predictions[0]  # 获取预测值

    return results


def predict_seat_allocation(flt_no, bd_type, cap, aircraft, duration, 
                            a, b, c, year, month, day, weekday, hour, minute, second, 
                            ab_price, bc_price, ac_price):

In [7]:
result = predict_seat_allocation(
    flt_no='P6p42wyZFEI=',
    bd_type='窄体',
    cap=132,
    aircraft='319',
    duration=2.38,
    a='tKjndGSl9NQ=',
    b='KETr2NAmAHE=',
    c='5t+HPO9Mu/w=',
    year=2023,
    month=1,
    day=1,
    weekday=6,
    hour=8,
    minute=50,
    second=0,
    ab_price=1000,
    bc_price=1000,
    ac_price=1500
)

print(result)

{'AB_PAX': 79.35475, 'BC_PAX': 62.66055, 'AC_PAX': 63.429333}


In [8]:
def allocate_seats(pax_predictions, ab_price, bc_price, ac_price, cap):
    """
    根据收益最大化策略，分配航段 AB, BC, AC 的客舱容量。
    
    输入:
    - pax_predictions: 字典，包含 AB, BC, AC 的乘客数预测值，例如 {'AB_PAX': 120, 'BC_PAX': 110, 'AC_PAX': 130}。
    - ab_price: float，航段 AB 的票价。
    - bc_price: float，航段 BC 的票价。
    - ac_price: float，航段 AC 的票价。
    - cap: int，总客舱容量。

    输出:
    - allocation: 字典，包含分配给 AB, BC, AC 的舱位数量。
    """
    # 获取航段预测乘客数
    ab_pax = round(pax_predictions['AB_PAX'])
    bc_pax = round(pax_predictions['BC_PAX'])
    ac_pax = round(pax_predictions['AC_PAX'])

    # 短途类（S 类）的最大容量
    s_max = max(ab_pax, bc_pax)
    s_min = min(ab_pax, bc_pax)

    # 分配结果初始化
    allocation = {'AB': 0, 'BC': 0, 'AC': 0}

    # 判断是否满座
    if s_max + ac_pax <= cap:
        # 未满座：按比例分配 S 类和 L 类
        total_pax = s_max + ac_pax
        s_cap = round(cap * (s_max / total_pax))  # S 类分配的容量
        l_cap = cap - s_cap  # L 类分配的容量

        # S 类：直接分配 s_cap（AB 和 BC 同时销售）
        allocation['AB'] = s_cap
        allocation['BC'] = s_cap

        # L 类：分配剩余容量
        allocation['AC'] = l_cap
    else:
        # 已满座：根据收益优先级分配
        s_revenue = ab_price + bc_price
        l_revenue = ac_price

        if s_revenue > l_revenue:
            # 优先分配给 S 类的 min(AB_PAX, BC_PAX)
            s_cap = min(s_min, cap)
            allocation['AB'] = s_cap
            allocation['BC'] = s_cap

            # 剩余容量分配给 L 类
            remaining_cap = cap - s_cap
            allocation['AC'] = min(ac_pax, remaining_cap)
        else:
            # 优先分配给 L 类
            allocation['AC'] = min(ac_pax, cap)  # L 类尽量满足 AC 的预测乘客数
            remaining_cap = cap - allocation['AC']  # 剩余容量分配给 S 类
            allocation['AB'] = remaining_cap
            allocation['BC'] = remaining_cap

    return allocation


In [19]:
# 示例输入
pax_predictions = {'AB_PAX': 20, 'BC_PAX': 20, 'AC_PAX': 30}
ab_price = 1047.58
bc_price = 950.32
ac_price = 1400.75
cap = 200

# 调用函数
allocation = allocate_seats(pax_predictions, ab_price, bc_price, ac_price, cap)

# 输出结果
print("舱位分配情况:", allocation)

舱位分配情况: {'AB': 80, 'BC': 80, 'AC': 120}


因为未满座，所以ab，bc，ac按比例分配舱位

In [9]:
# 示例输入
pax_predictions = {'AB_PAX': 120, 'BC_PAX': 110, 'AC_PAX': 130}
ab_price = 1047.58
bc_price = 950.32
ac_price = 1400.75
cap = 200

# 调用函数
allocation = allocate_seats(pax_predictions, ab_price, bc_price, ac_price, cap)

# 输出结果
print("舱位分配情况:", allocation)

舱位分配情况: {'AB': 110, 'BC': 110, 'AC': 90}


因为ab_price+bc_price>ac_price,所以优先分配舱位给短途

In [10]:
# 示例输入
pax_predictions = {'AB_PAX': 120, 'BC_PAX': 110, 'AC_PAX': 130}
ab_price = 1047.58
bc_price = 950.32
ac_price = 2400.75
cap = 200

# 调用函数
allocation = allocate_seats(pax_predictions, ab_price, bc_price, ac_price, cap)

# 输出结果
print("舱位分配情况:", allocation)

舱位分配情况: {'AB': 70, 'BC': 70, 'AC': 130}


因为ab_price+bc_price<ac_price,所以优先分配舱位给长途

## 求总收益最大值

求不同票价情况下的总收益最大值

相较于原来的情况的收益增加情况

In [11]:
import numpy as np
from scipy.optimize import minimize

In [12]:
def total_revenue(ab_price, bc_price, ac_price, pax_predictions, cap):
    """
    计算给定票价组合下的总收益
    """
    # 计算每个航段的乘客数
    ab_pax = pax_predictions['AB_PAX']
    bc_pax = pax_predictions['BC_PAX']
    ac_pax = pax_predictions['AC_PAX']
    
    # 调用航段舱位分配函数来获取最终分配的舱位
    allocation = allocate_seats(pax_predictions, ab_price, bc_price, ac_price, cap)

    # 计算总收益
    revenue = (allocation['AB'] * ab_price) + (allocation['BC'] * bc_price) + (allocation['AC'] * ac_price)
    return -revenue  # 因为优化器是最小化目标函数，所以返回负收益进行最大化

In [13]:
from tqdm import tqdm  # 导入 tqdm 库，用于显示进度条
import numpy as np

def optimize_ticket_prices(flt_no, bd_type, cap, aircraft, duration, a, b, c, year, month, day, weekday, hour, minute, second):
    """
    优化票价以最大化总收益
    
    输入:
    - flt_no: 航班号
    - bd_type: 机型类型
    - cap: 总客舱容量
    - aircraft: 航空器类型
    - duration: 航程时间
    - a, b, c: 其他航班特征
    - year, month, day, weekday, hour, minute, second: 时间特征

    输出:
    - 最优票价组合 (ab_price, bc_price, ac_price)
    """
    # 设置票价的网格范围
    price_range = (500, 1500)  # 设置票价范围
    step_size = 200  # 设置步长
    ab_price_vals = np.arange(price_range[0], price_range[1], step_size)
    bc_price_vals = np.arange(price_range[0], price_range[1], step_size)
    ac_price_vals = np.arange(price_range[0], price_range[1], step_size)
    
    # 用于保存最大收益的票价组合
    best_ab_price = None
    best_bc_price = None
    best_ac_price = None
    best_revenue = -np.inf  # 初始化最大收益为负无穷

    # 使用 tqdm 包装循环，显示进度条
    total_combinations = len(ab_price_vals) * len(bc_price_vals) * len(ac_price_vals)
    with tqdm(total=total_combinations, desc="优化票价中", unit="组合") as pbar:
        # 网格搜索所有票价组合
        for ab_price in ab_price_vals:
            for bc_price in bc_price_vals:
                for ac_price in ac_price_vals:
                    # 使用给定的航班特征和票价计算乘客预测数据
                    pax_predictions = predict_seat_allocation(
                        flt_no=flt_no,
                        bd_type=bd_type,
                        cap=cap,
                        aircraft=aircraft,
                        duration=duration,
                        a=a,
                        b=b,
                        c=c,
                        year=year,
                        month=month,
                        day=day,
                        weekday=weekday,
                        hour=hour,
                        minute=minute,
                        second=second,
                        ab_price=ab_price,
                        bc_price=bc_price,
                        ac_price=ac_price
                    )
                    
                    # 计算当前票价组合下的收益
                    revenue = -total_revenue(ab_price, bc_price, ac_price, pax_predictions, cap)
                    if revenue > best_revenue:
                        best_revenue = revenue
                        best_ab_price = ab_price
                        best_bc_price = bc_price
                        best_ac_price = ac_price
                    
                    # 更新进度条
                    pbar.update(1)

    return best_ab_price, best_bc_price, best_ac_price


In [14]:
# 示例输入
flt_no = 'Fssppujk3x0='
bd_type = '窄体'
cap = 120
aircraft = '320'
duration = 1.32
a = 'h9GisD/ZayE='
b = 'Lue5PP9SfQU='
c = 'tKjndGSl9NQ='
year = 2023
month = 1
day = 1
weekday = 6
hour = 8
minute = 50
second = 0

# 优化票价
best_ab_price, best_bc_price, best_ac_price = optimize_ticket_prices(
    flt_no, bd_type, cap, aircraft, duration, a, b, c, year, month, day, weekday, hour, minute, second
)

print(f"最优票价组合：AB票价={best_ab_price}, BC票价={best_bc_price}, AC票价={best_ac_price}")

# 使用最优票价预测客流量
pax_predictions = predict_seat_allocation(
    flt_no = 'Fssppujk3x0=',
    bd_type = '窄体',
    cap = 120,
    aircraft = '320',
    duration = 1.32,
    a = 'h9GisD/ZayE=',
    b = 'Lue5PP9SfQU=',
    c = 'tKjndGSl9NQ=',
    year = 2023,
    month = 1,
    day = 1,
    weekday = 6,
    hour = 8,
    minute = 50,
    second = 0,
    ab_price=best_ab_price,
    bc_price=best_bc_price,
    ac_price=best_ac_price
)

print(f"此时客流分配为: {pax_predictions}")

# 调用 allocate_seats 函数进行座位分配
allocation = allocate_seats(pax_predictions, best_ab_price, best_bc_price, best_ac_price, cap)

print(f"cap:{cap}")

# 输出分配结果
print("舱位分配情况:", allocation)

revenue = -total_revenue(best_ab_price, best_bc_price, best_ac_price, pax_predictions, cap)
print(f"此时总收益为{revenue}")

优化票价中: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 125/125 [00:22<00:00,  5.45组合/s]


最优票价组合：AB票价=1300, BC票价=1300, AC票价=1300
此时客流分配为: {'AB_PAX': 69.00776, 'BC_PAX': 62.30365, 'AC_PAX': 62.731094}
cap:120
舱位分配情况: {'AB': 62, 'BC': 62, 'AC': 58}
此时总收益为236600


In [18]:
# 使用最优票价预测客流量
pax_predictions = predict_seat_allocation(
    flt_no = 'Fssppujk3x0=',
    bd_type = '窄体',
    cap = 120,
    aircraft = '320',
    duration = 1.32,
    a = 'h9GisD/ZayE=',
    b = 'Lue5PP9SfQU=',
    c = 'tKjndGSl9NQ=',
    year = 2023,
    month = 1,
    day = 1,
    weekday = 6,
    hour = 8,
    minute = 50,
    second = 0,
    ab_price=300,
    bc_price=300,
    ac_price=300
)

print(f"此时客流分配为: {pax_predictions}")

# 调用 allocate_seats 函数进行座位分配
allocation = allocate_seats(pax_predictions, 300, 300, 300, cap)

print(f"cap:{cap}")

# 输出分配结果
print("舱位分配情况:", allocation)

revenue = -total_revenue(300, 300, 300, pax_predictions, cap)
print(f"此时总收益为{revenue}")

此时客流分配为: {'AB_PAX': 68.333595, 'BC_PAX': 59.499023, 'AC_PAX': 61.23534}
cap:120
舱位分配情况: {'AB': 59, 'BC': 59, 'AC': 61}
此时总收益为53700
